In [21]:
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import seaborn as sns
import os

BASE_DIR = os.getcwd()
DATA_PATH = os.path.join('..', 'data', 'avocado_processed_city.csv')
MODEL_PATH = os.path.join('..', 'models', 'xgb_avocado_LosAngeles_causal.pkl')

El modelo original al usar AVGPrice como feature genera un problema ya que aprende correlaciones historicas, pues aprende que cuando precio fue bajo ventas fueron altas pero esto no significa que bajar el precio cause mas ventas, puede ser que ambos sean efectos de un tercer factor externo

En el analisis exploratorio se detecto que en septiembre y octubre el precio promedio se dispara a 1.47 y 1.57 dolares mientras que el volumen de ventas baja a 1.3 millones y 1.15 millones aprox comparado con el promedio anual de 1.55 millones, el modelo ve esto como precio alto igual a ventas bajas y concluye que subir precio causa menos ventas cuando esto puede deberse a un factor externo como falta de stock en esa epoca del año, pues este efecto se repite en los 3 años analizados por lo que no es algo puntual, es tipico

El modelo causal soluciona esto al no usar AVGPrice como feature si no que usa la suma de ventas totales por fecha como un sustituto de la oferta disponible y scarcity_month que marca septiembre y octubre como meses de escasez,la ventaja es que que este modelo causal puede responder que pasa si cambio el precio manteniendo oferta constante mientras que el modelo original solo responde que paso historicamente cuando precio fue X

In [ ]:
model_original = joblib.load('../models/xgb_avocado_LosAngeles.pkl')
model_causal = joblib.load('../models/xgb_avocado_LosAngeles_causal.pkl')

df = pd.read_csv(DATA_PATH)
df = df[df['region'] == 'LosAngeles'].copy()
df['Date'] = pd.to_datetime(df['Date'])

#features causales
df['supply_total'] = df.groupby('Date')['Volumen'].transform('sum')
df['scarcity_month'] = df['month'].isin([9, 10]).astype(int)

test = df[df['Date'] >= '2017-01-01']

#prediccion original 
X_test_original = test[['AVGPrice', 'month', 'year', 'week', 'type_organic']]  # ← Agregado 'year'
preds_original = model_original.predict(X_test_original)

#prediccion causal
X_test_causal = test[['month', 'week', 'type_organic', 'supply_total', 'scarcity_month']]
preds_causal = model_causal.predict(X_test_causal)

print(f"MAE modelo original: {np.abs(test['Volumen'] - preds_original).mean():,.0f}")
print(f"MAE modelo causal: {np.abs(test['Volumen'] - preds_causal).mean():,.0f}")

fig = go.Figure()
fig.add_trace(go.Scatter(x=test['Date'], y=test['Volumen'],mode='lines', name='real',line=dict(color='black', width=2)))
fig.add_trace(go.Scatter(x=test['Date'], y=preds_original,mode='lines', name='original (precio como feature)',line=dict(color='steelblue', width=2)))
fig.add_trace(go.Scatter(x=test['Date'], y=preds_causal,mode='lines', name='causal (ventas totales por fecha + escasez)',line=dict(color='darkorange', width=2)))
fig.update_layout(
    title='volumen real vs predicho en ambos modelos para depues de 2017',
    xaxis_title='fecha',
    yaxis_title='volumen de ventas',
    hovermode='x unified',
    template='plotly_white',
    height=500
)
fig.show()

MAE modelo original: 32,707
MAE modelo causal: 12,301


El error absoluto medio MAE dio que el modelo original se equivoca en promedio 32,707 unidades por semana mientras que el modelo causal tiene un MAE de 12,301 unidades en promedio.

El modelo causal predice mejor que el original a pesar de no usar el precio como variable lo que demuestra que el precio no era lo que realmente explicaba el volumen de ventas sino que factores como la oferta disponible y la epoca del año los que determinaban cuanto se vendia.

Simular escenario con precio $1.57 en julio vs septiembre para el modelo causal y comprobar si el modelo causal aprendio queen sep/oct se vende menos porque hay menos oferta disponible y no porque sea septiembre ni porque el precio sea alto.

In [ ]:
supply_normal = df[df['scarcity_month'] == 0]['supply_total'].mean()

orig_julio = pd.DataFrame({'AVGPrice': [1.57], 'month': [7], 'year': [2018], 'week': [28], 'type_organic': [0]})
orig_sep   = pd.DataFrame({'AVGPrice': [1.57], 'month': [9], 'year': [2018], 'week': [36], 'type_organic': [0]})

print(f"Modelo ORIGINAL con precio $1.57:\n")
print(f"Julio (sin escasez): {model_original.predict(orig_julio)[0]:,.0f} unidades")
print(f"Septiembre (con escasez): {model_original.predict(orig_sep)[0]:,.0f} unidades")

causal_con_escasez = pd.DataFrame({'month': [9], 'week': [36], 'type_organic': [0], 'supply_total': [supply_normal], 'scarcity_month': [1]})
causal_sin_escasez = pd.DataFrame({'month': [9], 'week': [36], 'type_organic': [0], 'supply_total': [supply_normal], 'scarcity_month': [0]})

vol_con = model_causal.predict(causal_con_escasez)[0]
vol_sin = model_causal.predict(causal_sin_escasez)[0]

print("\nCausal en septiembre con oferta sin escasez y con escasez")
print(f"proxy de oferta {supply_normal:,.0f}:\n")
print(f"con escasez (scarcity_month=1): {vol_con:,.0f} unidades") #predice con escasez
print(f"sin escasez (scarcity_month=0): {vol_sin:,.0f} unidades") #predice que sin escasez,
print(f"\ndiferencia por escasez: {vol_con - vol_sin:+,.0f} unidades")

Modelo ORIGINAL con precio $1.57:

Julio (sin escasez): 2,246,272 unidades
Septiembre (con escasez): 2,097,567 unidades

Causal en septiembre con oferta sin escasez y con escasez
proxy de oferta 3,107,198:

con escasez (scarcity_month=1): 3,030,076 unidades
sin escasez (scarcity_month=0): 3,032,523 unidades

diferencia por escasez: -2,448 unidades


El modelo original con precio de $1.57 predice 2,246,272 unidades en julio y 2,097,076 para septiembre, una diferencia de 148,705 entre ambos meses con el mismo precio, el modelo distingue algo entre los meses por tener month como feature pero no puede saber si la caida en septiembre fue causada por el precio alto o por la escasez u otro factor externo, solo ve que historicamente ese precio ocurrio junto con ventas bajas y hace una correlacion

El modelo causal con el mismo nivel de ventas totales como proxy de oferta da resultados casi identicos con y sin escasez, 3,030,076 vs 3,032,523 unidades, esto indica que cuando la oferta disponible se mantiene constante en niveles normales, la escasez no cambia la prediccion porque es la variable supply_total la que esta capturando el efecto real, es decir la caida en ventas de septiembre y octubre se explica por la baja oferta de esos meses y no por el precio alto.

In [52]:
#oferta promedio de enero
enero_normal = df[(df['month'] == 1) & (df['year'].isin([2016, 2017]))]
supply_enero = enero_normal['supply_total'].mean()
print(f"la oferta promedio real en enero: {supply_enero:,.0f}")

test_original_120 = pd.DataFrame({
    'AVGPrice': [1.20], 'month': [1], 'year': [2018], 'week': [1], 'type_organic': [0]  # ← Agregado 'year'
})
test_original_090 = pd.DataFrame({
    'AVGPrice': [0.90], 'month': [1], 'year': [2018], 'week': [1], 'type_organic': [0]  # ← Agregado 'year'
})

vol_original_120 = model_original.predict(test_original_120)[0]
vol_original_090 = model_original.predict(test_original_090)[0]

print("\nmodelo original")
print(f"precio $1.20 = {vol_original_120:,.0f} unidades")
print(f"precio $0.90 = {vol_original_090:,.0f} unidades")
print(f"diferencia: {vol_original_090 - vol_original_120:+,.0f} unidades")

test_causal = pd.DataFrame({'month': [1], 'week': [1], 'type_organic': [0],'supply_total': [supply_enero],'scarcity_month': [0]}) # oferta constante
vol_causal = model_causal.predict(test_causal)[0]

print("\nmodelo causal")
print(f"volumen : {vol_causal:,.0f} unidades")

la oferta promedio real en enero: 3,578,982

modelo original
precio $1.20 = 2,470,046 unidades
precio $0.90 = 2,879,460 unidades
diferencia: +409,414 unidades

modelo causal
volumen : 3,471,436 unidades


El modelo original predice 2,470,046 unidades con precio $1.20 y 2,879,460 unidades con precio $0.90 lo que sugiere que bajar el precio en enero generaria casi medio millon de unidades extra, pero esto es una correlacion historica no una relacion causal, el modelo aprendio que precio bajo coincidio con ventas altas en el pasado y reproduce esa asociacion

el modelo causal con la oferta promedio real de enero de 3,578,982 unidades predice un volumen de 3,471,436 unidades sin importar el precio que se fije, porque el precio no es una variable del modelo, lo que determina el volumen es cuanto stock hay disponible, como en el caso de sept-oct

la diferencia entre ambos modelos es relevante pues si se usa el modelo original para decidir precios se podria concluir que bajar de $1.20 a $0.90 vale la pena porque supuestamente genera 409,414 unidades mas, el modelo causal dice que esa ganancia en volumen no existe, que en enero con oferta normal ya hay una demanda dada y recortar precio solo reduce el margen sin aumentar ventas reales

simulacion de escenarios extremos, enero con oferta alta vs septiembre con oferta baja

In [53]:
escenario_alta = pd.DataFrame({
    'month': [1] * 100,
    'week': [1] * 100,
    'type_organic': [0] * 100,
    'supply_total': [supply_enero * 1.2] * 100,  # 20% mas oferta
    'scarcity_month': [0] * 100
})

supply_sept = df[df['month'] == 9]['supply_total'].mean()
escenario_baja = pd.DataFrame({
    'month': [9] * 100,
    'week': [36] * 100,
    'type_organic': [0] * 100,
    'supply_total': [supply_sept * 0.8] * 100,  # 20% menos oferta
    'scarcity_month': [1] * 100
})

vol_alta = model_causal.predict(escenario_alta)[0]
vol_baja = model_causal.predict(escenario_baja)[0]

print(f"oferta alta (enero +20%): {vol_alta:,.0f} unidades")
print(f"oferta baja (sept -20%): {vol_baja:,.0f} unidades")
print(f"diferencia: {vol_alta - vol_baja:+,.0f} unidades ({((vol_alta - vol_baja)/vol_baja)*100:.1f}%)")

oferta alta (enero +20%): 4,216,564 unidades
oferta baja (sept -20%): 1,969,654 unidades
diferencia: +2,246,910 unidades (114.1%)


esta simulacion muestra el rango de demanda que el modelo causal puede predecir segun el nivel de oferta disponible, con enero en 20% por encima del promedio el modelo predice 4,216,564 unidades, con septiembre en 20% por debajo del promedio de ese mes el modelo predice 1,969,654 unidades, una diferencia de 2,246,910 unidades que equivale al 114% mas de ventas en el escenario favorable

esto confirma que la oferta disponible es el principal determinante del volumen de ventas para el modelo no el precio, un mes con abundante stock puede mover el doble de unidades que un mes con stock bajo independientemente de lo que cueste la palta

para aumentar ventas no hay que bajar el precio sino asegurarse de tener stock disponible, especialmente en los meses de septiembre y octubre donde la caida historica se explica por restricciones de oferta y no por el precio alto que se observa en esos meses, mas bien el precio alto se debe a las restricciones de oferta